# Debug: Sky Background & Dark Current Discrepancy

Analyse détaillée pour comprendre pourquoi les évolutions avec sky et dark current ne matchent pas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, 'notebooks')

from Observation import Observation, load_instruments
from astropy.stats import signal_to_noise_oir_ccd

plt.style.use('seaborn-v0_8-darkgrid')
print("✓ Imports OK")

In [ ]:
# Load instruments
instruments, database = load_instruments()

# Use GALEX FUV (no read noise = simpler)
instrument_name = "GALEX FUV"

# Extract parameters
params = {}
for i, charact in enumerate(instruments['Charact.']):
    if charact and not isinstance(charact, np.ma.core.MaskedConstant):
        value = instruments[instrument_name][i]
        if not isinstance(value, np.ma.core.MaskedConstant):
            params[charact] = value

print(f"Instrument: {instrument_name}")
print(f"\nKey parameters:")
print(f"  Signal: {params['Signal']:.2e} erg/cm²/s/arcsec²/Å")
print(f"  Sky: {params['Sky']:.2e} erg/cm²/s/arcsec²/Å")
print(f"  Dark current: {params['Dark_current']:.4f} e⁻/pix/hour")
print(f"  Read noise: {params['RN']:.2f} e⁻")
print(f"  Collecting area: {params['Collecting_area']:.3f} m²")
print(f"  Throughput: {params['Throughput']:.3f}")
print(f"  QE: {params['QE']:.3f}")
print(f"  Atmosphere: {params['Atmosphere']:.3f}")
print(f"  Pixel scale: {params['pixel_scale']:.2f} arcsec/pix")
print(f"  Dispersion: {params['dispersion']:.2f} Å/pix")
print(f"  Wavelength: {params['wavelength']:.1f} nm")

## 1. Analyse détaillée d'un cas simple

Comparons les valeurs intermédiaires pour un cas de référence.

In [ ]:
# Reference case
t_exp = 1000.0  # seconds

# Generic ETC
obs = Observation(
    instruments=instruments,
    instrument=instrument_name,
    exposure_time=t_exp,
    SNR_res="per pix",
    IFS=False,
    test=True
)

print("="*70)
print("GENERIC ETC - Internal values")
print("="*70)
print(f"SNR: {obs.SNR[obs.i]:.6f}")
print(f"\nPixel info:")
print(f"  pixels_total_source: {obs.pixels_total_source}")
print(f"  elem_size: {obs.elem_size}")
print(f"  number_pixels_used: {obs.number_pixels_used}")
print(f"  flux_fraction: {obs.flux_fraction:.6f}")

print(f"\nConversion factors:")
print(f"  factor_CU2el: {obs.factor_CU2el:.6e}")
print(f"  factor_CU2el_sky: {obs.factor_CU2el_sky:.6e}")
print(f"  Ratio (sky/signal): {obs.factor_CU2el_sky/obs.factor_CU2el:.4f}")

print(f"\nSignal (e⁻/pix/exp):")
print(f"  Signal_el (Generic): {obs.Signal_el[obs.i]:.6e}")
print(f"  Signal_el * pixels_total_source: {obs.Signal_el[obs.i] * obs.pixels_total_source:.6e}")

print(f"\nNoise components (e⁻):")
print(f"  sky (electrons): {obs.sky[obs.i]:.6e}")
print(f"  Sky_noise (bruit): {obs.Sky_noise[obs.i]:.6e}")
print(f"  sky * pixels_total_source: {obs.sky[obs.i] * obs.pixels_total_source:.6e}")
print(f"  Dark_current_f (e⁻): {obs.Dark_current_f[obs.i]:.6e}")
print(f"  Total_noise_final: {obs.Total_noise_final[obs.i]:.6e}")

print(f"\nSize info:")
print(f"  source_size_arcsec_after_slit: {obs.source_size_arcsec_after_slit} arcsec²")
print(f"  slit_size_arcsec_after_slit: {obs.slit_size_arcsec_after_slit} arcsec²")
print(f"  pixel_scale²: {params['pixel_scale']**2} arcsec²")

In [ ]:
# Astropy calculation - step by step
print("="*70)
print("ASTROPY SNR - Step by step calculation")
print("="*70)

# Step 1: Convert flux to photons
wavelength_nm = params['wavelength']
E_photon = 1.986e-8 / wavelength_nm  # erg
print(f"\n1. Photon energy at {wavelength_nm} nm: {E_photon:.4e} erg")

signal_flux = params['Signal']  # erg/cm²/s/arcsec²/Å
sky_flux = params['Sky']        # erg/cm²/s/arcsec²/Å

signal_photons_rate = signal_flux / E_photon  # photons/cm²/s/arcsec²/Å
sky_photons_rate = sky_flux / E_photon        # photons/cm²/s/arcsec²/Å

print(f"\n2. Flux to photons:")
print(f"   Signal: {signal_flux:.2e} erg → {signal_photons_rate:.4e} photons/cm²/s/arcsec²/Å")
print(f"   Sky: {sky_flux:.2e} erg → {sky_photons_rate:.4e} photons/cm²/s/arcsec²/Å")

# Step 2: Apply telescope/instrument
collecting_area_cm2 = params['Collecting_area'] * 1e4  # m² → cm²
pixel_area_arcsec2 = params['pixel_scale']**2          # arcsec²
wavelength_range = params['dispersion']                 # Å/pixel
total_throughput = params['Throughput'] * params['QE'] * params['Atmosphere']

print(f"\n3. Telescope/instrument:")
print(f"   Collecting area: {collecting_area_cm2:.1f} cm²")
print(f"   Pixel area: {pixel_area_arcsec2:.2f} arcsec²")
print(f"   Wavelength range: {wavelength_range:.2f} Å/pix")
print(f"   Total throughput: {total_throughput:.4f}")

# Step 3: Calculate electron rates PER PIXEL
source_eps = (signal_photons_rate * 
              collecting_area_cm2 * 
              pixel_area_arcsec2 * 
              wavelength_range * 
              total_throughput)

sky_eps = (sky_photons_rate * 
           collecting_area_cm2 * 
           pixel_area_arcsec2 * 
           wavelength_range * 
           total_throughput)

dark_eps = params['Dark_current'] / 3600.0  # e⁻/pix/hour → e⁻/s/pix

print(f"\n4. Electron rates (PER PIXEL):")
print(f"   source_eps: {source_eps:.6e} e⁻/s/pix")
print(f"   sky_eps: {sky_eps:.6e} e⁻/s/pix")
print(f"   dark_eps: {dark_eps:.6e} e⁻/s/pix")

# Step 4: Total electrons over exposure (per pixel)
n_pixels = 1  # per pixel mode

signal_total = source_eps * t_exp
sky_total = sky_eps * t_exp * n_pixels
dark_total = dark_eps * t_exp * n_pixels
read_total = params['RN']**2 * n_pixels

print(f"\n5. Total electrons (t_exp = {t_exp}s, npix = {n_pixels}):")
print(f"   Signal: {signal_total:.6e} e⁻")
print(f"   Sky: {sky_total:.6e} e⁻")
print(f"   Dark: {dark_total:.6e} e⁻")
print(f"   Read²: {read_total:.6e} e⁻²")

# Step 5: SNR calculation
noise_variance = signal_total + sky_total + dark_total + read_total
noise_total = np.sqrt(noise_variance)
snr_manual = signal_total / noise_total

print(f"\n6. SNR calculation:")
print(f"   Noise variance: {noise_variance:.6e}")
print(f"   Noise (sqrt): {noise_total:.6e} e⁻")
print(f"   SNR (manual): {snr_manual:.6e}")

# Compare with Astropy function
snr_astropy = signal_to_noise_oir_ccd(
    t=t_exp,
    source_eps=source_eps,
    sky_eps=sky_eps,
    dark_eps=dark_eps,
    rd=params['RN'],
    npix=n_pixels,
    gain=1.0
)
print(f"   SNR (astropy): {snr_astropy:.6e}")

## 2. Comparaison Signal vs Sky

Regardons les ratios signal/sky pour comprendre la discrepancy.

In [ ]:
print("="*70)
print("COMPARISON: Generic ETC vs Astropy")
print("="*70)

print(f"\nGeneric ETC (BRUT):")
print(f"  Signal: {obs.Signal_el[obs.i]:.6e} e⁻")
print(f"  Sky: {obs.sky[obs.i]:.6e} e⁻")
print(f"  Dark: {obs.Dark_current_f[obs.i]:.6e} e⁻")
print(f"  SNR: {obs.SNR[obs.i]:.6e}")

print(f"\nGeneric ETC (CORRIGE × pixels_total_source = {obs.pixels_total_source:.2f}):")
signal_corrected = obs.Signal_el[obs.i] * obs.pixels_total_source
sky_corrected = obs.sky[obs.i] * obs.pixels_total_source
print(f"  Signal: {signal_corrected:.6e} e⁻")
print(f"  Sky: {sky_corrected:.6e} e⁻")
print(f"  Dark: {obs.Dark_current_f[obs.i]:.6e} e⁻")

print(f"\nAstropy (per pixel):")
print(f"  Signal: {signal_total:.6e} e⁻")
print(f"  Sky: {sky_total:.6e} e⁻")
print(f"  Dark: {dark_total:.6e} e⁻")
print(f"  SNR: {snr_astropy:.6e}")

print(f"\nRatios (Generic BRUT / Astropy):")
print(f"  Signal: {obs.Signal_el[obs.i]/signal_total:.4f}")
print(f"  Sky: {obs.sky[obs.i]/sky_total:.4f}")
print(f"  Dark: {obs.Dark_current_f[obs.i]/dark_total:.4f}")
print(f"  SNR: {obs.SNR[obs.i]/snr_astropy:.4f}")

print(f"\nRatios (Generic CORRIGE / Astropy):")
print(f"  Signal: {signal_corrected/signal_total:.4f}")
print(f"  Sky: {sky_corrected/sky_total:.4f}")
print(f"  Dark: {obs.Dark_current_f[obs.i]/dark_total:.4f}")

print(f"\n⚠️ Key observation:")
if abs(signal_corrected/signal_total - 1) < 0.15:
    print(f"  ✓ Signal OK après correction!")
else:
    print(f"  ✗ Signal differs significantly even after correction: {signal_corrected/signal_total:.4f}")
    
if abs(sky_corrected/sky_total - 1) < 0.15:
    print(f"  ✓ Sky OK après correction!")
else:
    print(f"  ✗ Sky differs significantly even after correction: {sky_corrected/sky_total:.4f}")

## 3. Test avec variation du Sky (analyser si pixels_total_source change)

**HYPOTHESE**: Si les variations sont différentes, c'est peut-être parce que pixels_total_source ou les facteurs changent avec les paramètres.

In [ ]:
# Test sky variation with more points
sky_factors = np.logspace(-2, 2, 30)  # 0.01x to 100x nominal

sky_flux_nominal = params['Sky']
signal_flux = params['Signal']
exposure_time = params.get('exposure_time', 1000.0)

data = []

for factor in sky_factors:
    sky_flux = sky_flux_nominal * factor
    
    # Modify instrument
    original_sky = instruments[instrument_name][list(instruments['Charact.']).index('Sky')]
    instruments[instrument_name][list(instruments['Charact.']).index('Sky')] = sky_flux
    
    # Generic ETC
    obs = Observation(
        instruments=instruments,
        instrument=instrument_name,
        exposure_time=exposure_time,
        SNR_res="per pix",
        IFS=False,
        test=True
    )
    snr_generic = obs.SNR[obs.i]
    signal_generic = obs.Signal_el[obs.i]
    sky_generic = obs.sky[obs.i]
    dark_generic = obs.Dark_current_f[obs.i]
    noise_generic = obs.Total_noise_final[obs.i]
    
    # CORRECTION: multiplier par pixels_total_source
    signal_generic_corrected = signal_generic * obs.pixels_total_source
    sky_generic_corrected = sky_generic * obs.pixels_total_source
    
    # Facteurs
    factor_CU2el = obs.factor_CU2el
    factor_CU2el_sky = obs.factor_CU2el_sky
    pixels_total_source = obs.pixels_total_source
    
    # Restore
    instruments[instrument_name][list(instruments['Charact.']).index('Sky')] = original_sky
    
    # Astropy
    sky_photons = sky_flux / E_photon
    sky_eps = (sky_photons * collecting_area_cm2 * pixel_area_arcsec2 * 
               wavelength_range * total_throughput)
    
    snr_astropy = signal_to_noise_oir_ccd(
        t=exposure_time,
        source_eps=source_eps,
        sky_eps=sky_eps,
        dark_eps=dark_eps,
        rd=params['RN'],
        npix=1,
        gain=1.0
    )
    
    sky_astropy = sky_eps * exposure_time
    
    data.append({
        'sky_factor': factor,
        'sky_flux': sky_flux,
        'snr_generic': snr_generic,
        'snr_astropy': snr_astropy,
        'rel_diff_pct': 100 * (snr_generic - snr_astropy) / snr_astropy,
        'signal_generic': signal_generic,
        'signal_generic_corrected': signal_generic_corrected,
        'sky_generic': sky_generic,
        'sky_generic_corrected': sky_generic_corrected,
        'sky_astropy': sky_astropy,
        'sky_ratio_brut': sky_generic / sky_astropy if sky_astropy > 0 else np.nan,
        'sky_ratio_corrected': sky_generic_corrected / sky_astropy if sky_astropy > 0 else np.nan,
        'dark_generic': dark_generic,
        'noise_generic': noise_generic,
        'factor_CU2el': factor_CU2el,
        'factor_CU2el_sky': factor_CU2el_sky,
        'pixels_total_source': pixels_total_source,
    })

df_sky = pd.DataFrame(data)
print(f"Sky variation test: {len(df_sky)} points")
df_sky.head(10)

In [ ]:
# DIAGNOSTIC: Est-ce que pixels_total_source change avec sky?
print("="*70)
print("DIAGNOSTIC: pixels_total_source variation avec sky")
print("="*70)
print(f"Min pixels_total_source: {df_sky['pixels_total_source'].min()}")
print(f"Max pixels_total_source: {df_sky['pixels_total_source'].max()}")
print(f"Variation: {(df_sky['pixels_total_source'].max() - df_sky['pixels_total_source'].min()) / df_sky['pixels_total_source'].mean() * 100:.2f}%")

print(f"\nfactor_CU2el variation:")
print(f"Min: {df_sky['factor_CU2el'].min():.6e}")
print(f"Max: {df_sky['factor_CU2el'].max():.6e}")
print(f"Variation: {(df_sky['factor_CU2el'].max() - df_sky['factor_CU2el'].min()) / df_sky['factor_CU2el'].mean() * 100:.2f}%")

print(f"\nfactor_CU2el_sky variation:")
print(f"Min: {df_sky['factor_CU2el_sky'].min():.6e}")
print(f"Max: {df_sky['factor_CU2el_sky'].max():.6e}")
print(f"Variation: {(df_sky['factor_CU2el_sky'].max() - df_sky['factor_CU2el_sky'].min()) / df_sky['factor_CU2el_sky'].mean() * 100:.2f}%")

In [ ]:
# Plot sky comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# SNR comparison
ax = axes[0, 0]
ax.loglog(df_sky['sky_factor'], df_sky['snr_generic'], 'o-', label='Generic ETC')
ax.loglog(df_sky['sky_factor'], df_sky['snr_astropy'], 's--', label='Astropy', alpha=0.7)
ax.set_xlabel('Sky Factor')
ax.set_ylabel('SNR')
ax.set_title('SNR vs Sky Factor')
ax.legend()
ax.grid(True, alpha=0.3)

# Relative difference SNR
ax = axes[0, 1]
ax.semilogx(df_sky['sky_factor'], df_sky['rel_diff_pct'], 'o-', color='red')
ax.axhline(0, color='black', linestyle='--')
ax.axhline(15, color='orange', linestyle=':')
ax.axhline(-15, color='orange', linestyle=':')
ax.set_xlabel('Sky Factor')
ax.set_ylabel('Relative Difference (%)')
ax.set_title('SNR Relative Difference vs Sky Factor')
ax.grid(True, alpha=0.3)

# pixels_total_source variation
ax = axes[0, 2]
ax.semilogx(df_sky['sky_factor'], df_sky['pixels_total_source'], 'o-', color='purple')
ax.set_xlabel('Sky Factor')
ax.set_ylabel('pixels_total_source')
ax.set_title('pixels_total_source vs Sky Factor')
ax.grid(True, alpha=0.3)

# Sky electrons comparison (brut)
ax = axes[1, 0]
ax.loglog(df_sky['sky_factor'], df_sky['sky_generic'], 'o-', label='Generic ETC (brut)', color='red')
ax.loglog(df_sky['sky_factor'], df_sky['sky_astropy'], 's--', label='Astropy', alpha=0.7, color='blue')
ax.set_xlabel('Sky Factor')
ax.set_ylabel('Sky electrons')
ax.set_title('Sky Electrons (BRUT) Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

# Sky electrons comparison (corrected)
ax = axes[1, 1]
ax.loglog(df_sky['sky_factor'], df_sky['sky_generic_corrected'], 'o-', label='Generic ETC (×npix)', color='green')
ax.loglog(df_sky['sky_factor'], df_sky['sky_astropy'], 's--', label='Astropy', alpha=0.7, color='blue')
ax.set_xlabel('Sky Factor')
ax.set_ylabel('Sky electrons')
ax.set_title('Sky Electrons (CORRECTED) Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

# Sky ratio (corrected)
ax = axes[1, 2]
ax.semilogx(df_sky['sky_factor'], df_sky['sky_ratio_corrected'], 'o-', color='green', label='Corrected')
ax.semilogx(df_sky['sky_factor'], df_sky['sky_ratio_brut'], 's--', color='red', alpha=0.5, label='Brut')
ax.axhline(1, color='black', linestyle='--')
ax.set_xlabel('Sky Factor')
ax.set_ylabel('Sky Ratio (Generic/Astropy)')
ax.set_title('Sky Electrons Ratio')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nSky ratio (Generic corrected/Astropy): {df_sky['sky_ratio_corrected'].mean():.4f} ± {df_sky['sky_ratio_corrected'].std():.4f}")

## 4. Test avec variation du Dark Current

In [ ]:
# Test dark current variation
dark_factors = np.logspace(-2, 2, 30)  # 0.01x to 100x nominal

dark_current_nominal = params['Dark_current']

data_dark = []

for factor in dark_factors:
    dark_current = dark_current_nominal * factor
    
    # Modify instrument
    original_dc = instruments[instrument_name][list(instruments['Charact.']).index('Dark_current')]
    instruments[instrument_name][list(instruments['Charact.']).index('Dark_current')] = dark_current
    
    # Generic ETC
    obs = Observation(
        instruments=instruments,
        instrument=instrument_name,
        exposure_time=exposure_time,
        SNR_res="per pix",
        IFS=False,
        test=True
    )
    snr_generic = obs.SNR[obs.i]
    dark_generic = obs.Dark_current_f[obs.i]
    
    # Facteurs
    pixels_total_source = obs.pixels_total_source
    
    # Restore
    instruments[instrument_name][list(instruments['Charact.']).index('Dark_current')] = original_dc
    
    # Astropy
    dark_eps_var = dark_current / 3600.0
    
    snr_astropy = signal_to_noise_oir_ccd(
        t=exposure_time,
        source_eps=source_eps,
        sky_eps=sky_eps,
        dark_eps=dark_eps_var,
        rd=params['RN'],
        npix=1,
        gain=1.0
    )
    
    dark_astropy = dark_eps_var * exposure_time
    
    data_dark.append({
        'dark_factor': factor,
        'dark_current': dark_current,
        'snr_generic': snr_generic,
        'snr_astropy': snr_astropy,
        'rel_diff_pct': 100 * (snr_generic - snr_astropy) / snr_astropy,
        'dark_generic': dark_generic,
        'dark_astropy': dark_astropy,
        'dark_ratio': dark_generic / dark_astropy if dark_astropy > 0 else np.nan,
        'pixels_total_source': pixels_total_source,
    })

df_dark = pd.DataFrame(data_dark)
print(f"Dark current variation test: {len(df_dark)} points")
df_dark.head(10)

In [ ]:
# DIAGNOSTIC: Est-ce que pixels_total_source change avec dark?
print("="*70)
print("DIAGNOSTIC: pixels_total_source variation avec dark")
print("="*70)
print(f"Min pixels_total_source: {df_dark['pixels_total_source'].min()}")
print(f"Max pixels_total_source: {df_dark['pixels_total_source'].max()}")
print(f"Variation: {(df_dark['pixels_total_source'].max() - df_dark['pixels_total_source'].min()) / df_dark['pixels_total_source'].mean() * 100:.2f}%")

In [ ]:
# Plot dark comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# SNR comparison
ax = axes[0, 0]
ax.loglog(df_dark['dark_factor'], df_dark['snr_generic'], 'o-', label='Generic ETC')
ax.loglog(df_dark['dark_factor'], df_dark['snr_astropy'], 's--', label='Astropy', alpha=0.7)
ax.set_xlabel('Dark Factor')
ax.set_ylabel('SNR')
ax.set_title('SNR vs Dark Factor')
ax.legend()
ax.grid(True, alpha=0.3)

# Relative difference
ax = axes[0, 1]
ax.semilogx(df_dark['dark_factor'], df_dark['rel_diff_pct'], 'o-', color='red')
ax.axhline(0, color='black', linestyle='--')
ax.axhline(15, color='orange', linestyle=':')
ax.axhline(-15, color='orange', linestyle=':')
ax.set_xlabel('Dark Factor')
ax.set_ylabel('Relative Difference (%)')
ax.set_title('Relative Difference vs Dark Factor')
ax.grid(True, alpha=0.3)

# Dark electrons comparison
ax = axes[1, 0]
ax.loglog(df_dark['dark_factor'], df_dark['dark_generic'], 'o-', label='Generic ETC dark (e⁻)')
ax.loglog(df_dark['dark_factor'], df_dark['dark_astropy'], 's--', label='Astropy dark (e⁻)', alpha=0.7)
ax.set_xlabel('Dark Factor')
ax.set_ylabel('Dark electrons')
ax.set_title('Dark Electrons Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

# Dark ratio
ax = axes[1, 1]
ax.semilogx(df_dark['dark_factor'], df_dark['dark_ratio'], 'o-', color='purple')
ax.axhline(1, color='black', linestyle='--')
ax.set_xlabel('Dark Factor')
ax.set_ylabel('Dark Ratio (Generic/Astropy)')
ax.set_title('Dark Electrons Ratio')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nDark ratio (Generic/Astropy): {df_dark['dark_ratio'].mean():.4f} ± {df_dark['dark_ratio'].std():.4f}")

## 5. Export CSV pour analyse

In [ ]:
# Save to CSV
df_sky.to_csv('./debug_sky_variation.csv', index=False)
df_dark.to_csv('./debug_dark_variation.csv', index=False)

print("✓ Saved debug_sky_variation.csv")
print("✓ Saved debug_dark_variation.csv")

# Summary
print(f"\n{'='*70}")
print("SUMMARY")
print(f"{'='*70}")
print(f"\nSky test:")
print(f"  Sky ratio brut (Generic/Astropy): {df_sky['sky_ratio_brut'].mean():.4f}")
print(f"  Sky ratio corrected (Generic×npix/Astropy): {df_sky['sky_ratio_corrected'].mean():.4f}")
print(f"  Max rel diff SNR: {df_sky['rel_diff_pct'].abs().max():.2f}%")
print(f"  pixels_total_source variation: {(df_sky['pixels_total_source'].max() - df_sky['pixels_total_source'].min()) / df_sky['pixels_total_source'].mean() * 100:.2f}%")

print(f"\nDark test:")
print(f"  Dark ratio (Generic/Astropy): {df_dark['dark_ratio'].mean():.4f}")
print(f"  Max rel diff SNR: {df_dark['rel_diff_pct'].abs().max():.2f}%")
print(f"  pixels_total_source variation: {(df_dark['pixels_total_source'].max() - df_dark['pixels_total_source'].min()) / df_dark['pixels_total_source'].mean() * 100:.2f}%")

## 6. Analyse des variations des facteurs de conversion

Si les variations sont différentes entre sky et dark, c'est peut-être parce que:
1. `pixels_total_source` change avec les paramètres
2. `factor_CU2el` ou `factor_CU2el_sky` changent
3. Un autre terme non-linéaire dans le calcul du SNR

In [ ]:
# Plot factor variations for sky
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.semilogx(df_sky['sky_factor'], df_sky['factor_CU2el'], 'o-', label='factor_CU2el')
ax.semilogx(df_sky['sky_factor'], df_sky['factor_CU2el_sky'], 's-', label='factor_CU2el_sky')
ax.set_xlabel('Sky Factor')
ax.set_ylabel('Conversion factor')
ax.set_title('Conversion factors vs Sky variation')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.semilogx(df_sky['sky_factor'], df_sky['factor_CU2el_sky'] / df_sky['factor_CU2el'], 'o-', color='purple')
ax.set_xlabel('Sky Factor')
ax.set_ylabel('Ratio factor_sky / factor_signal')
ax.set_title('Ratio of conversion factors vs Sky variation')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Ratio factor_sky/factor_signal: {(df_sky['factor_CU2el_sky'] / df_sky['factor_CU2el']).mean():.4f}")